# AgentReach AI — Phase 3: Read one user-approved Google Doc

This phase builds on the Phase 2 Drive search and allows a **single Google Doc selected by you** to be previewed as text. The Needle tool is restricted to the one approved file ID, requires your confirmation flag, and returns at most 5,000 characters.

## Authorization and privacy notes
- Run cells in order. Google authorization happens in your Colab account and may grant notebook code access beyond this one tool. Review the Google prompt and the notebook code before approving.
- Do not choose a document containing passwords, access tokens, financial-account details, private records, or information you do not want placed in notebook output.
- This notebook reads only Google Docs documents. It does not write, share, move, or delete files and does not read PDFs, Sheets, or arbitrary Drive files.
- The agent cannot choose a different document: the tool checks the exact file ID that you selected and approved in a separate cell.
- Needle is a small tool-calling model, not a general chat model. The test verifies that it invokes the approved preview tool and returns its result.

In [ ]:
%pip install -q cactus-needle google-api-python-client

## 2. Import dependencies

In [ ]:
import json
import sys
import needle
from google.colab import auth
from googleapiclient.discovery import build

print('Python:', sys.version.split()[0])
print('Needle import: OK')

## 3. Authorize Google Drive

If your current runtime is already authorized, the account may be reused. Review Google's prompt before approving anything.

In [ ]:
auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Drive client initialized.')

## 4. List recent Google Docs by name only

This cell lists up to 20 Google Docs and their IDs so you can choose one. It does not download content.

In [ ]:
GOOGLE_DOCS_MIME = 'application/vnd.google-apps.document'
doc_listing = drive_service.files().list(
    q=f"mimeType = '{GOOGLE_DOCS_MIME}' and trashed = false",
    pageSize=20,
    orderBy='modifiedTime desc',
    fields='files(id,name,mimeType,modifiedTime,webViewLink)',
).execute()
docs = doc_listing.get('files', [])
if not docs:
    print('No Google Docs were returned. Check account access or place a test document in Drive.')
else:
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. {doc.get('name', '(untitled)')} | ID: {doc['id']} | Modified: {doc.get('modifiedTime', 'unknown')}")
    print('\nChoose only a document you are comfortable previewing in notebook output.')

## 5. Select one exact document and confirm reading

Paste the ID from the list above. Then type the exact confirmation phrase when prompted. If you do not confirm, the content-reading tool remains disabled.

In [ ]:
available_doc_ids = {doc['id'] for doc in docs}
selected_file_id = input('Paste the ID of one listed Google Doc (or type SKIP): ').strip()
if selected_file_id == 'SKIP':
    approved_file_id = None
    approved_to_read = False
    print('Skipped. No document content will be read.')
elif selected_file_id not in available_doc_ids:
    approved_file_id = None
    approved_to_read = False
    print('That ID is not in the list from this cell. Re-run this cell and choose a listed ID.')
else:
    chosen_doc = next(doc for doc in docs if doc['id'] == selected_file_id)
    print('Selected document:', chosen_doc.get('name', '(untitled)'))
    phrase = input("To approve reading up to 5,000 characters, type READ THIS DOCUMENT: ").strip()
    approved_to_read = (phrase == 'READ THIS DOCUMENT')
    approved_file_id = selected_file_id if approved_to_read else None
    print('Document read approved for this exact file ID.' if approved_to_read else 'Not approved. No content will be read.')

## 6. Define a restricted content-preview tool

The function checks the approved file ID, verifies that it is a Google Doc, asks the Drive API for a text export, and truncates the output to 5,000 characters.

In [ ]:
@needle.tool(triggers=[r'(?i)\bapproved Google Doc\b', r'(?i)\bpreview the selected document\b'])
def preview_approved_google_doc(file_id: str) -> dict:
    """Preview up to 5,000 characters from the single Google Doc the user selected and explicitly approved in this notebook. Requires the exact approved file ID."""
    if not approved_to_read or not approved_file_id:
        return {'ok': False, 'error': 'No document has been explicitly approved for reading.'}
    if not isinstance(file_id, str) or file_id != approved_file_id:
        return {'ok': False, 'error': 'Access denied: use only the exact user-approved Google Doc ID.'}
    try:
        meta = drive_service.files().get(
            fileId=approved_file_id,
            fields='id,name,mimeType,capabilities(canDownload)',
        ).execute()
        if meta.get('mimeType') != GOOGLE_DOCS_MIME:
            return {'ok': False, 'error': 'Only native Google Docs can be previewed in this phase.'}
        if meta.get('capabilities', {}).get('canDownload') is False:
            return {'ok': False, 'error': 'Google reports that this document cannot be downloaded/exported.'}
        raw = drive_service.files().export(
            fileId=approved_file_id,
            mimeType='text/plain',
        ).execute()
        text = raw.decode('utf-8', errors='replace') if isinstance(raw, bytes) else str(raw)
        limit = 5000
        return {
            'ok': True,
            'name': meta.get('name'),
            'file_id': approved_file_id,
            'preview_characters': min(len(text), limit),
            'truncated': len(text) > limit,
            'preview': text[:limit],
            'note': 'Read-only preview of the single user-approved Google Doc.',
        }
    except Exception as exc:
        return {'ok': False, 'error': f'Document preview failed: {type(exc).__name__}'}

print('Preview tool defined. Approved file ID present:', bool(approved_file_id))

## 7. Test the agent on the approved document

In [ ]:
if not approved_to_read or not approved_file_id:
    print('Skipped because no document was approved.')
else:
    agent = needle.Needle(tools=[preview_approved_google_doc])
    result = agent.run(
        'Preview the approved Google Doc with ID ' + approved_file_id +
        '. Call only the preview_approved_google_doc tool for this exact approved Google Doc.'
    )
    print('Tool call result:')
    print(json.dumps(result, indent=2, ensure_ascii=False, default=str))

## Success checkpoint

- The document list shows names and IDs only.
- You explicitly select one listed Google Doc and approve reading it.
- The preview tool rejects any other file ID and returns no more than 5,000 characters.
- No write operations are present.

If the agent returns no tool call, inspect the `function_calls`, `confidence`, and `results` fields. Needle is designed primarily for tool calling and can return an empty call list for unsupported requests. The direct preview function can be tested after approval with `preview_approved_google_doc(approved_file_id)`, but do not paste sensitive document content into chat.